# Keratoconus four-map image training
Attach the labelled image dataset and the small `kaggle_pipeline` code dataset, enable a GPU, then run the cells below. Start with EfficientNetV2-S before launching the larger models.

In [ ]:
!pip install -q timm grad-cam

In [ ]:
from pathlib import Path

manifest = next(Path('/kaggle/input').glob('**/metadata/eye_manifest.csv'))
DATA_ROOT = manifest.parent.parent
TRAIN_SCRIPT = next(Path('/kaggle/input').glob('**/kaggle_pipeline/train.py'))
PIPELINE = TRAIN_SCRIPT.parent
print('Dataset:', DATA_ROOT)
print('Pipeline:', PIPELINE)

In [ ]:
# Choose: efficientnetv2_s, convnext_b, swin_b, dinov2, or retfound
ENCODER = 'efficientnetv2_s'
FOLD = 1
EPOCHS = 25
BATCH_SIZE = 4

In [ ]:
import subprocess, sys

command = [
    sys.executable, str(TRAIN_SCRIPT),
    '--data-root', str(DATA_ROOT),
    '--encoder', ENCODER,
    '--output-dir', '/kaggle/working/keratoconus_models',
    '--fold', str(FOLD),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--grad-accumulation', '2',
]
# RETFound additionally needs:
# command += ['--retfound-checkpoint', '/kaggle/input/.../RETFound_cfp_weights.pth']
subprocess.run(command, cwd=PIPELINE, check=True)

In [ ]:
import json

run_dir = Path('/kaggle/working/keratoconus_models') / ENCODER / f'fold_{FOLD}'
print(json.dumps(json.loads((run_dir / 'metrics.json').read_text()), indent=2))
print('Download checkpoint:', run_dir / 'best_model.pth')